# Pilot: QLoRA Harness adapter (Term 1)

Fine-tunes a small open code model on the **verified** harness attempts that ProofHound logged in
`training/data/harness_attempts.jsonl` (records with `reached = true`), then measures first-attempt
success on held-out records.

Run on Google Colab with a GPU runtime (T4 is enough for a 1.5B model).

Rules:
- Records whose `split` is `eval` are never used for training (benchmark leakage).
- The base model is chosen in the literature review; set `BASE_MODEL` below.

In [ ]:
!pip -q install "transformers>=4.46" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.44" "datasets>=3.0" accelerate

In [ ]:
# Upload training/data/harness_attempts.jsonl to the Colab session (or mount Google Drive).
DATA = "harness_attempts.jsonl"
BASE_MODEL = ""          # e.g. a 1.5B-7B open code model chosen in the literature review
OUTPUT_DIR = "harness-adapter"
assert BASE_MODEL, "Set BASE_MODEL first"


In [ ]:
import json, random
from datasets import Dataset

records = [json.loads(l) for l in open(DATA, encoding="utf-8") if l.strip()]
usable = [r for r in records if r["split"] != "eval" and r["reached"]]
print(f"{len(records)} attempts, {len(usable)} verified and allowed for training")

random.seed(0)
random.shuffle(usable)
cut = max(1, int(len(usable) * 0.9))
train = Dataset.from_list([{"prompt": r["prompt"], "completion": r["output"]} for r in usable[:cut]])
held_out = usable[cut:]
print(len(train), "train /", len(held_out), "held out")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules="all-linear")
args = SFTConfig(
    output_dir=OUTPUT_DIR, num_train_epochs=3, per_device_train_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=2e-4, logging_steps=5,
    save_strategy="epoch", max_length=4096, fp16=True, report_to=[],
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train,
                     processing_class=tokenizer, peft_config=lora)
trainer.train()
trainer.save_model(OUTPUT_DIR)

## Evaluate
Generate a harness for each held-out prompt and save them to `held_out_outputs.jsonl`.
Copy that file back to the project; ProofHound's checks (syntax / load / reach / coverage) score
first-attempt success, so the score comes from execution, not from this notebook.

In [ ]:
model.eval()
with open("held_out_outputs.jsonl", "w", encoding="utf-8") as f:
    for r in held_out:
        ids = tokenizer(r["prompt"], return_tensors="pt").to(model.device)
        out = model.generate(**ids, max_new_tokens=1024, do_sample=False)
        text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        f.write(json.dumps({"finding_id": r["finding_id"], "case": r["case"], "output": text}) + "
")
print("wrote", len(held_out))